# SkinCanserClassification

Bu projede deri kanserini algilayan bir CNN modeli gelistirecegiz. Model kaydedip streamlit uygulamasina cevirecegiz ve huggingface'de calisir hale getirecegiz.

In [17]:
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv2D,Dense, Flatten, Input, MaxPooling2D, Dropout, BatchNormalization, Reshape
import os

In [22]:
labels=['Cancer','Non_Cancer']

In [23]:
ls

 Volume in drive C has no label.
 Volume Serial Number is 08D6-2CCE

 Directory of C:\Users\kosey\OneDrive\Documents\GitHub\ai_course\Day12

09.11.2025  10:27    <DIR>          .
09.11.2025  06:59    <DIR>          ..
09.11.2025  10:13    <DIR>          .ipynb_checkpoints
09.11.2025  09:32             3.074 app.py
09.11.2025  08:12           141.824 cars.xls
09.11.2025  09:32            51.005 cars.xlsx
09.11.2025  10:04    <DIR>          Skin_Data
09.11.2025  10:27             4.698 SkinCanserClassification.ipynb
               4 File(s)        200.601 bytes
               4 Dir(s)  387.137.196.032 bytes free


In [24]:
img_path='Skin_Data/'

In [25]:
img_list=[]
label_list=[]
for label in labels:
    for img_file in os.listdir(img_path+label):
        img_list.append(img_path+label+"/"+img_file)
        label_list.append(label)

In [26]:
df=pd.DataFrame({'img':img_list,'label':label_list})

In [27]:
df

,img,label
0,Skin_Data/Cancer/1007-1.jpg,Cancer
1,Skin_Data/Cancer/1010-01.JPG,Cancer
2,Skin_Data/Cancer/1012-2.JPG,Cancer
3,Skin_Data/Cancer/1031-1.jpg,Cancer
4,Skin_Data/Cancer/1051-3(94).jpg,Cancer
...,...,...
283,Skin_Data/Non_Cancer/953-1.JPG,Non_Cancer
284,Skin_Data/Non_Cancer/954-3.JPG,Non_Cancer
285,Skin_Data/Non_Cancer/955.JPG,Non_Cancer
286,Skin_Data/Non_Cancer/984.JPG,Non_Cancer


In [29]:
d=({'Cancer':1, 'Non_Cancer':0})
df['label_encoded']=df['label'].map(d)

In [30]:
df

,img,label,label_encoded
0,Skin_Data/Cancer/1007-1.jpg,Cancer,1
1,Skin_Data/Cancer/1010-01.JPG,Cancer,1
2,Skin_Data/Cancer/1012-2.JPG,Cancer,1
3,Skin_Data/Cancer/1031-1.jpg,Cancer,1
4,Skin_Data/Cancer/1051-3(94).jpg,Cancer,1
...,...,...,...
283,Skin_Data/Non_Cancer/953-1.JPG,Non_Cancer,0
284,Skin_Data/Non_Cancer/954-3.JPG,Non_Cancer,0
285,Skin_Data/Non_Cancer/955.JPG,Non_Cancer,0
286,Skin_Data/Non_Cancer/984.JPG,Non_Cancer,0


In [32]:
x=[]
for img in df['img']:
    img=cv2.imread(str(img))
    img=cv2.resize(img,(170,170))
    img=img/255.0 # normalize et
    x.append(img)

In [33]:
x=np.array(x)

In [34]:
x.shape

(288, 170, 170, 3)

In [35]:
x

array([[[[0.69411765, 0.74901961, 0.95686275],
         [0.6745098 , 0.72941176, 0.9372549 ],
         [0.59607843, 0.64313725, 0.8627451 ],
         ...,
         [0.49411765, 0.62745098, 0.84705882],
         [0.49411765, 0.63137255, 0.85098039],
         [0.51372549, 0.64313725, 0.8627451 ]],

        [[0.63921569, 0.69411765, 0.88627451],
         [0.64313725, 0.69803922, 0.90196078],
         [0.63529412, 0.69803922, 0.90588235],
         ...,
         [0.52156863, 0.63137255, 0.84705882],
         [0.5254902 , 0.63921569, 0.85490196],
         [0.5372549 , 0.64313725, 0.8627451 ]],

        [[0.64313725, 0.70588235, 0.89803922],
         [0.65882353, 0.71372549, 0.91764706],
         [0.64313725, 0.70980392, 0.91372549],
         ...,
         [0.54117647, 0.64705882, 0.86666667],
         [0.56078431, 0.66666667, 0.88627451],
         [0.5372549 , 0.64705882, 0.8627451 ]],

        ...,

        [[0.64313725, 0.74117647, 0.9372549 ],
         [0.62745098, 0.72156863, 0.91764706]

In [46]:
y=df['label_encoded']

In [47]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.20, random_state=42)

In [48]:
y_train=np.array(y_train, dtype=np.int32)
y_test=np.array(y_test, dtype=np.int32)

In [49]:
model=Sequential()
model.add(Input(shape=(170, 170, 3)))
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))  # Added activation function
model.add(Dense(1, activation='sigmoid'))  # Ensure 2 classes for binary classification

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [50]:
history=model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=20,verbose=1)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 272ms/step - accuracy: 0.5696 - loss: 2.5446 - val_accuracy: 0.7414 - val_loss: 0.6747
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 222ms/step - accuracy: 0.6043 - loss: 0.6870 - val_accuracy: 0.7414 - val_loss: 0.6070
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 234ms/step - accuracy: 0.7087 - loss: 0.6624 - val_accuracy: 0.7414 - val_loss: 0.6125
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 224ms/step - accuracy: 0.7087 - loss: 0.5823 - val_accuracy: 0.7414 - val_loss: 0.5743
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 236ms/step - accuracy: 0.7348 - loss: 0.5453 - val_accuracy: 0.7069 - val_loss: 0.6153
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 232ms/step - accuracy: 0.7739 - loss: 0.5110 - val_accuracy: 0.8103 - val_loss: 0.4823
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 231ms/step - accuracy: 0.8348 - loss: 0.4144 - val_accuracy: 0.7414 - val_loss: 0.5323
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 246ms/step - accuracy: 0.8522 - loss: 0.3616 - val_accuracy: 0.8448 - val_loss:

In [ ]:
model.save('skin_cancer_model.h5')

In [53]:
model.save('skin_cancer_model.keras')

# Transfer Learning

Akilli insan aklini kullanir. Dahada akilli insan basklarinin aklini da kullanandir.

In [55]:
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [56]:
data_dir='Skin_Data'
img_width,img_heigth=224,224
train_datagen=ImageDataGenerator(rescale=1/255, validation_split=0.20)
train_datagenerator=train_datagen.flow_from_directory(directory=data_dir,target_size=(img_width,img_heigth),
                                class_mode='binary', subset='training')
test_datagen=ImageDataGenerator(rescale=1/255)
test_datagenerator=train_datagen.flow_from_directory(directory=data_dir,target_size=(img_width,img_heigth),
                                class_mode='binary', subset='validation')

# VGG16 hazir modeli cekiliyor
base_model=VGG16(weights='imagenet', input_shape=(img_width,img_heigth,3), include_top=False)

# Sequential yeni model olusturuluyor ve VGG16 modeli eklenoyor
model=Sequential()
model.add(base_model)
# hazir alinan VGG16 model icin ogrenmeyi kapatiyoruzki ogrenilmis model bozulmasin
for layer in base_model.layers:
    layer.trainable=False

# hazir ogrenilmis modelin ustune bizim goreseller ile tekrardan ogrenme yaptiriyoruz
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model.fit(train_datagenerator,epochs=10,validation_data=test_datagenerator)

Found 232 images belonging to 2 classes.
Found 56 images belonging to 2 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Epoch 1/10


C:\Users\kosey\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.6207 - loss: 2.3288 - val_accuracy: 0.8571 - val_loss: 0.3488
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 23s 3s/step - accuracy: 0.8664 - loss: 0.3091 - val_accuracy: 0.8214 - val_loss: 0.3400
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9440 - loss: 0.1607 - val_accuracy: 0.8393 - val_loss: 0.3653
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9741 - loss: 0.1057 - val_accuracy: 0.8214 - val_loss: 0.3469
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - accuracy: 0.9914 - loss: 0.0779 - val_accuracy: 0.8214 - val_loss: 0.3313
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9871 - loss: 0.0667 - val_accuracy: 0.8214 - val_loss: 0.3336
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9871 - loss: 0.0680 - val_accuracy: 0.8393 - val_loss: 0.3800
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9828 - loss: 0.0556 - val_accuracy: 0.8393 - val_loss: 0.4025
Epoch 9/10
8/8 ━━━━

In [57]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1024)           │    25,691,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,791,173 (350.16 MB)

 Trainable params: 25,692,161 (98.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

 Optimizer params: 51,384,324 (196.02 MB)

In [58]:
model.save('tl_skin_cancer_model.h5')

In [59]:
model.save('tl_skin_cancer_model.keras')